# [LES - kEqn] NACA4412

## Preamble

In [1]:
# Standard Library
import sys
import os
from pathlib import Path
import foamnordic as fno
import numpy as np
import matplotlib.pyplot as plt
import onsaemiro as osm

### Directory & Path

In [2]:
# FoamNordic Project Directory
PROJECT_DIR = Path("/scratch/<allocation-account>/<user>")
CASE_TYPE = "les"
CASE_NAME = "NACA4412"
BASE_DIR = PROJECT_DIR / "Codes" / "FoamNordic"
MAIN_DIR = BASE_DIR / "tutorials" / "foamnordic_tutorials" / "incompressible"
OF_SCRIPT_DIR = BASE_DIR / "tutorials" / "openfoam_tutorials" / "incompressible" / CASE_TYPE / CASE_NAME

# Output Directory
OUTPUT_DIR = MAIN_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### Configuration

In [3]:
# DataGraph Configuration
FIG_X, FIG_Y = 3.5, 2.55
FIGURE_SIZE = (FIG_X, FIG_Y)
PALETTE = osm.get_palette("OKABE_ITO")
osm.set_style(figure_size=FIGURE_SIZE)

In [ ]:
# HPC configuration
ACCOUNT = "<allocation-account>" # e.g. project_xxxxxxx
PARTITION = "small"
TIME = "00:15:00"
N_NODES = 1
N_TASKS = 16
CPUS_PER_TASK = 1
MEM_PER_CPU = "2G"

# FoamNordic Slurm configuration
of_scheduler = fno.Slurm.openfoam(
    nodes=N_NODES,
    ntasks=N_TASKS,
    cpus_per_task=CPUS_PER_TASK,
    mem_per_cpu=MEM_PER_CPU,
)

model_scheduler = fno.Slurm.model(
    cpus_per_task=1,
    mem_per_cpu=MEM_PER_CPU
)

scheduler = fno.Slurm(
    account=ACCOUNT,
    partition=PARTITION,
    time=TIME,
    openfoam=of_scheduler,
    model=model_scheduler
)

In [5]:
# FoamNordic configuration
SEED = 42
key = fno.Random.key(seed=SEED, scope="global")

## Example - Closure Modelling

### K Equation Closure

In [6]:
# kEqn function for subgrid-scale turbulence modeling
def keqn_function(k, velocity_grad, delta, C_k=0.094, C_e=1.048):
    k_positive = fno.Math.maximum(k, 0.0)
    sqrt_k = fno.Math.sqrt(k_positive)

    eddy_viscosity = C_k * delta * sqrt_k

    dev_two_symm_grad = fno.Math.dev(2.0 * fno.Math.symm(velocity_grad))
    k_production = eddy_viscosity * fno.Math.ddot(velocity_grad, dev_two_symm_grad)
    k_dissipation_coeff = C_e * sqrt_k / delta

    return eddy_viscosity, k_production, k_dissipation_coeff

In [7]:
# Define kEQN Closure
keqn_closure = fno.Closure(
    name="kEqnFjord",
    operator=fno.Operator.function(keqn_function),
    inputs={
        "k": fno.Field("k"),
        "velocity_grad": fno.Field.grad("U"),
        "delta": fno.Field.delta(),
    },
    outputs={
        "eddy_viscosity": fno.Field("nut"),
        "K_production": fno.Field("kProduction"),
        "K_dissipation_coeff": fno.Field("kDissipationCoeff"),
    },
    key=key
)

### Case Definition

In [8]:
# Initialize the OpenFOAM case
case = fno.OpenFOAM.Case(
    name=CASE_NAME,
    case_dir=OF_SCRIPT_DIR,
    run_dir=OUTPUT_DIR,
    of_cmd="module load openfoam/2512",
    shell="bash",
    application="pimpleFoam",
)

case.initialize(ranks=N_TASKS, mesh="blockMesh", validate_mesh=True)

### Submit Job

In [9]:
# Connect the OpenFOAM case and SLURM scheduler (launch a longship instance)
longship = fno.Longship(case=case, scheduler=scheduler, closures=(keqn_closure,))

# Set sail for the OpenFOAM case (submit the job to the HPC cluster)
run = longship.launch(start_timeout=900)

In [10]:
# Wait for the job to complete (polling the job status)
result = run.stop(force=False, timeout=3600, progress=True)

In [11]:
# Summary of the job result
result.summary(style="compact");

Job ID,Name,Status,Partition,Node,Elapsed
870074,NACA4412,succeeded,small,rc4133,00:01:08


### Postprocessing

In [12]:
# Postprocessing
post = result.postprocess

velocity = post.field("U", time_idx=-1)
pressure = post.field("p", time_idx=-1)

print("U shape:", velocity.shape)
print("p shape:", pressure.shape)

statistics = post.statistics(
    ["U", "p", "nut"],
    time_idx=-1,
    verbose=True,
)

U shape: (89728, 3)
p shape: (89728,)


Field,Min,Max,Mean,Std,RMS
U,1.193457e-06,1.820307e+00,8.267743e-01,4.611357e-01,9.466794e-01
p,-1.260086e+00,5.414912e-01,-2.109534e-01,3.377280e-01,3.981979e-01
nut,2.200743e-07,5.923383e-04,7.611088e-06,3.587457e-05,3.667306e-05
